# dx-local v2 — E1+E2 on combined v1(49)+v2(260) corpus
Pools the three `processed_*.csv` (v1 DDXPlus) with the three `v2_*.csv` (generated corpus), globally dedups signatures, re-splits deterministically 80/10/10 per pathology, then trains
E1 logreg + E2 LightGBM (pinned hyperparams), stack meta-learner,
temperature scaling, Mondrian conformal, ask_table.
Outputs `artifacts.zip`.


In [ ]:
import ast, hashlib, json, math, os, time, zipfile, zlib
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression

SEED = 20260822
IN = '/kaggle/input'
FILES_V1 = ('processed_train.csv', 'processed_validate.csv', 'processed_test.csv')
FILES_V2 = ('v2_train.csv', 'v2_validate.csv', 'v2_test.csv')

def _locate():
    wanted = {'processed_train.csv'} | {f for f in FILES_V2}
    for dp, _, fs in os.walk(IN):
        if wanted <= set(fs):
            return dp
    return None

DDX = None
for attempt in range(60):
    DDX = _locate()
    if DDX:
        break
    if attempt == 0:
        print('waiting for dataset mount; saw:', os.listdir(IN))
    time.sleep(10)
if DDX is None:
    raise RuntimeError(f'dataset never mounted; IN={os.listdir(IN)}')
print('dataset dir:', DDX)


In [ ]:
# ---- pool, dedup, deterministic stratified re-split ----
cols = ['PATHOLOGY', 'AGE', 'SEX', 'EVIDENCES']
frames = []
for fname in (*FILES_V1, *FILES_V2):
    df = pd.read_csv(f'{DDX}/{fname}', dtype={'PATHOLOGY': 'string'})[cols]
    frames.append(df)
    print(fname, len(df))
pool = pd.concat(frames, ignore_index=True)
del frames
pool['AGE'] = pool['AGE'].fillna(-1).astype(int)
key = (pool['PATHOLOGY'].astype(str) + '|' + pool['AGE'].astype(str) + '|'
       + pool['SEX'].astype(str) + '|' + pool['EVIDENCES'].astype(str))
before = len(pool)
pool = pool.loc[~key.duplicated()].reset_index(drop=True)
print(f'pooled={before} deduped={len(pool)}')

parts = {'train': [], 'validate': [], 'test': []}
for pathology, group in pool.groupby('PATHOLOGY', sort=False):
    gseed = (SEED + zlib.crc32(str(pathology).encode())) % 2**32
    idx = np.random.default_rng(gseed).permutation(len(group))
    n = len(group); c1, c2 = int(n*0.8), int(n*0.9)
    parts['train'].append(group.iloc[np.sort(idx[:c1])])
    parts['validate'].append(group.iloc[np.sort(idx[c1:c2])])
    parts['test'].append(group.iloc[np.sort(idx[c2:])])
del pool
splits = {}
for s, lst in parts.items():
    df = pd.concat(lst, ignore_index=True); del lst
    df['EVIDENCES'] = [tuple(sorted(ast.literal_eval(c or '[]')))
                       for c in df['EVIDENCES'].fillna('[]')]
    splits[s] = df
print({k: len(v) for k, v in splits.items()})


In [ ]:
def build_vocab(df):
    codes = sorted({c for row in df['EVIDENCES'] for c in row})
    return {'evidence_codes': tuple(codes),
            'pathologies': tuple(sorted(df['PATHOLOGY'].unique()))}

vocab = build_vocab(splits['train'])
classes = vocab['pathologies']; lut = {c: i for i, c in enumerate(classes)}
print('classes:', len(classes), 'codes:', len(vocab['evidence_codes']))

def design_matrix(df, vocab):
    index = {c: i for i, c in enumerate(vocab['evidence_codes'])}
    n_cols = len(index) + 3
    ages = df['AGE'].to_numpy(); sexes = df['SEX'].to_numpy()
    indptr = np.zeros(len(df) + 1, dtype=np.int64)
    col_chunks, val_chunks = [], []
    for r, evs in enumerate(df['EVIDENCES']):
        cols_ = [index[c] for c in evs if c in index]
        vals = [1.0] * len(cols_)
        cols_.append(len(index)); vals.append(float(ages[r]) if ages[r] == ages[r] else 0.0)
        cols_.append(len(index) + 1); vals.append(1.0 if sexes[r] == 'M' else 0.0)
        cols_.append(len(index) + 2); vals.append(1.0 if sexes[r] == 'F' else 0.0)
        col_chunks.append(np.asarray(cols_, dtype=np.int32))
        val_chunks.append(np.asarray(vals, dtype=np.float64))
        indptr[r + 1] = indptr[r] + len(cols_)
    indices = np.concatenate(col_chunks).astype(np.int32)
    data = np.concatenate(val_chunks)
    return csr_matrix((data, indices, indptr.astype(np.int32)), shape=(len(df), n_cols))

import gc
X, y = {}, {}
for s in ('train', 'validate', 'test'):
    y[s] = np.array([lut[p] for p in splits[s]['PATHOLOGY']])
    X[s] = design_matrix(splits[s], vocab)
    print(s, X[s].shape, 'nnz:', X[s].nnz)
train_patho = splits['train']['PATHOLOGY'].to_numpy()
train_evs = splits['train']['EVIDENCES'].to_numpy()
gc.collect()


In [ ]:
e1 = LogisticRegression(C=10.0, max_iter=500, random_state=SEED).fit(X['train'], y['train'])
print('E1 acc validate=%.4f test=%.4f' % (
    e1.score(X['validate'], y['validate']), e1.score(X['test'], y['test'])))


In [ ]:
params = dict(objective='multiclass', learning_rate=0.1, num_leaves=7,
              min_data_in_leaf=2, seed=SEED, deterministic=True,
              force_col_wise=True, num_threads=4, verbose=-1,
              num_class=len(classes))
e2 = lgb.train(params, lgb.Dataset(X['train'], label=y['train']), num_boost_round=150)
for s in ('validate', 'test'):
    acc = float((e2.predict(X[s]).argmax(1) == y[s]).mean())
    print(f'E2 acc {s}={acc:.4f}')


In [ ]:
records = []
for rid, (pathology, evs) in enumerate(zip(train_patho, train_evs)):
    bases = {c.split('_@_')[0] for c in evs}
    records.extend((rid, lut[pathology], b) for b in sorted(bases))
ex = pd.DataFrame(records, columns=['rid', 'cls', 'base'])
counts = ex.drop_duplicates(['rid', 'base']).groupby(['base', 'cls']).size()
totals = pd.Series(train_patho).map(lut).value_counts()
ask_table = {b: {i: (int(counts.get((b, i), 0)) / int(totals.get(i, 0))
                     if int(totals.get(i, 0)) else 0.0)
                 for i in range(len(classes))}
             for b in vocab['evidence_codes']}
print('ask_table bases:', len(ask_table))


In [ ]:
# ---- calibration: temperature scaling + stack + conformal ----
def softmax(z):
    z = z - z.max(1, keepdims=True)
    q = np.exp(z); return q / q.sum(1, keepdims=True)

def apply_T(p, T):
    return softmax(np.log(np.clip(p, 1e-12, None)) / T)

def nll(logT, p, yy):
    T = np.exp(logT[0]); q = apply_T(p, T)
    return -np.log(q[np.arange(len(yy)), yy] + 1e-12).sum()

def fit_T(p, yy):
    r = minimize(nll, [0.0], args=(p, yy), method='L-BFGS-B', bounds=[(-5, 5)])
    return float(np.exp(r.x[0]))

p1v_raw = e1.predict_proba(X['validate']); p2v_raw = e2.predict(X['validate'])
t1 = fit_T(p1v_raw, y['validate']); t2 = fit_T(p2v_raw, y['validate'])
print('T1=%.4f T2=%.4f' % (t1, t2))
p1v = apply_T(p1v_raw, t1); p2v = apply_T(p2v_raw, t2)
meta = LogisticRegression(C=1.0, max_iter=500).fit(np.hstack([p1v, p2v]), y['validate'])
fused_v = meta.predict_proba(np.hstack([p1v, p2v]))
scores = 1.0 - fused_v[np.arange(len(y['validate'])), y['validate']]
alpha = 0.05; n = len(scores)
marginal_q = float(np.sort(scores)[min(math.ceil((n + 1) * (1 - alpha)), n) - 1])
conf_q = {}
for c in range(len(classes)):
    s_ = scores[y['validate'] == c]
    conf_q[c] = float(np.sort(s_)[min(math.ceil((len(s_) + 1) * (1 - alpha)), len(s_)) - 1]) \
        if len(s_) >= 10 else marginal_q

# held-out fused metrics
p1tv = apply_T(e1.predict_proba(X['test']), t1)
p2tv = apply_T(e2.predict(X['test']), t2)
fused_t = meta.predict_proba(np.hstack([p1tv, p2tv]))
acc = float((fused_t.argmax(1) == y['test']).mean())
confidences = fused_t.max(1)
correct = (fused_t.argmax(1) == y['test']).astype(np.float64)
bins = np.minimum((confidences * 15).astype(int), 14)
ece_val = float(sum((bins == b).sum()/len(y['test']) *
                    abs(confidences[bins == b].mean() - correct[bins == b].mean())
                    for b in range(15) if (bins == b).any()))
try:
    from sklearn.metrics import f1_score
    mf1 = float(f1_score(y['test'], fused_t.argmax(1), average='macro'))
except Exception:
    mf1 = -1.0
print('STACK test acc=%.4f macro-F1=%.4f ECE=%.4f' % (acc, mf1, ece_val))
metrics = {'stack_acc': acc, 'macro_f1': mf1, 'ece': ece_val, 't1': t1, 't2': t2}


In [ ]:
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
out = '/kaggle/working/artifacts/'
json.dump({'evidence_codes': list(vocab['evidence_codes']),
           'pathologies': list(classes)}, open(out + 'vocab.json', 'w'))
joblib.dump(e1, out + 'e1_logreg.joblib')
e2.save_model(out + 'e2_lgbm.txt')
json.dump(ask_table, open(out + 'ask_table.json', 'w'))
json.dump({'t1': t1, 't2': t2, 'conformal': {str(k): v for k, v in conf_q.items()},
           'marginal': marginal_q}, open(out + 'calib.json', 'w'))
joblib.dump(meta, out + 'stack.joblib')
train_counts = pd.Series(train_patho).value_counts()
json.dump({c: int(train_counts.get(c, 0)) for c in classes},
          open(out + 'class_counts.json', 'w'))
json.dump({'n_classes': len(classes), **metrics},
          open(out + 'metrics.json', 'w'))
manifest = {}
for f in sorted(os.listdir(out)):
    h = hashlib.sha256(open(out + f, 'rb').read()).hexdigest()
    manifest[f] = h
json.dump(manifest, open(out + 'MANIFEST.sha256.json', 'w'), indent=1)
with zipfile.ZipFile('/kaggle/working/artifacts.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(out)):
        zf.write(out + f, f)
print('artifacts.zip written:', os.path.getsize('/kaggle/working/artifacts.zip')//1024//1024, 'MB')
